# ChartQA on a Colab TPU: replay a measured repair experiment

Run **Gemma 4 E2B-it** with EvalRX on a frozen 64-question ChartQA CONFIRM split, with no judge key.

The completed reference search selected `conditional_arithmetic_gate` from 13 candidates on EXPLORE. On CONFIRM, baseline and candidate both scored **32/64 (50%)**, with **9 repairs and 9 regressions**, verdict **`no_effect`**. No accuracy repair validated. Model weights are unchanged.

`reference_output.json` includes the exact candidate, pinned data, all original outputs and provenance. This notebook regenerates the answers; it does not simply display the saved scores. Use a fresh **TPU** runtime and a checkout containing these tutorial files. Unpublished files must be uploaded first; cloning cannot retrieve uncommitted changes.


In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path('/content/evalrx')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'ruinan', 'https://github.com/evalvitals/evalrx.git', str(repo)], check=True)
if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True)
else:
    print('Using an uploaded source checkout; see the reference JSON for source hashes.')
subprocess.run(['bash', str(repo / 'tools/colab/setup.sh')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(Path.home() / '.cache/evalrx/colab/constraints.txt'), '-e', str(repo) + '[data,contract,viz,stats]'], check=True)
env = os.environ.copy()
env.update(JAX_PLATFORMS='tpu,cpu', OMP_NUM_THREADS='2', TF_CPP_MIN_LOG_LEVEL='2')


## Replay the frozen candidate

The script verifies the pinned ChartQA images, loads Gemma directly on the TPU, regenerates the baseline and applies the frozen candidate to all 64 CONFIRM questions. Initial loading and compilation take minutes. Run in a subprocess so the notebook kernel does not retain TPU buffers.

Replaying the same questions checks reproducibility; it is not another independent confirmation experiment. Add `--limit 4` only for a quick four-question installation check.


In [ ]:
output = repo / 'examples/colab/outputs/replay.json'
subprocess.run([sys.executable, '-u', str(repo / 'examples/colab/replay_chartqa_repair.py'), '--reference', str(repo / 'examples/colab/reference_output.json'), '--output', str(output)], cwd=repo, env=env, check=True)


In [ ]:
import json
result = json.loads(output.read_text())
assert result['status'] == 'completed', result
v = result['fix']['attempted'][0]
print({k: v[k] for k in ['n_pairs', 'n_baseline_correct', 'n_candidate_correct', 'n_fixed', 'n_broken', 'verdict', 'e_value', 'e_threshold']})
for row in result['predictions']:
    print(row['id'], 'gold:', row['expected'], 'baseline:', row['observed'], 'candidate:', row['repaired'])


In [ ]:
from google.colab import files
files.download(str(output))


## Recorded native TPU output

The replay script was executed directly on the Colab TPU, without a judge or HTTP worker. Actual final console line:

```text
CONFIRM n=64: baseline=32, repaired=32, fixed=9, broken=9, verdict=no_effect
```

All 64 baseline answers and all 64 candidate answers exactly matched the original search run. See `reference_replay.json` and `reference_replay_console.txt` for the full output. Notebook code cells were syntax-checked; this recorded output comes from the script, not an execution of this notebook.


## Optional: run automatic diagnosis and repair search

This is a separate experiment, requiring an installed and authenticated judge CLI. The reference used Claude on a separate controller with all target-model inference on the TPU. See the README for the optional private worker and SSH forwarding workflow. No reference repair is supplied to the search.

The pipeline runs baseline, pinned M1 probes, structured M2/M3, M4, and automatic repair search up to L2. Free-form code generation and pre-fix surgery are disabled. Search uses EXPLORE and freezes one candidate for CONFIRM. A completed run may legitimately report no validated repair.


In [ ]:
RUN_SEARCH = False  # Requires an installed, authenticated Claude CLI in this runtime.
if RUN_SEARCH:
    subprocess.run([sys.executable, '-u', str(repo / 'examples/colab/run_chartqa_repair.py'), '--judge-provider', 'claude', '--judge-model', 'claude-sonnet-4-6', '--judge-effort', 'high', '--run-tag', 'my_search'], cwd=repo, env=env, check=True)
